# Experiment 2: CoT prompting

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# ── INSTALL DEPENDENCIES ─────────────────────────────────────────
!pip install -q transformers accelerate datasets sacrebleu evaluate \
    sentencepiece protobuf bitsandbytes peft trl

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — switch runtime!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ── GLOBAL CONFIG ────────────────────────────────────────────────

LANG_PAIRS = [
    ('eng_Latn', 'fra_Latn', 'English', 'French'),   # High resource
    ('fra_Latn', 'eng_Latn', 'French', 'English'),   # Reverse
    ('eng_Latn', 'xho_Latn', 'English', 'Xhosa'),    # Low resource
]

import json, re, random, pandas as pd, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm.notebook import tqdm

random.seed(42)
np.random.seed(42)

In [ ]:
# ── UTILITY FUNCTIONS (batched / parallelized generation) ─────────────

def load_flores(src_lang, tgt_lang, split='devtest', n=50):
    src_ds = load_dataset('openlanguagedata/flores_plus', src_lang, split=split)
    tgt_ds = load_dataset('openlanguagedata/flores_plus', tgt_lang, split=split)
    return [ex['text'] for ex in src_ds][:n], [ex['text'] for ex in tgt_ds][:n]

# Quick test
test_src, test_ref = load_flores('eng_Latn', 'fra_Latn', split='devtest', n=3)
for s, r in zip(test_src, test_ref):
    print(f"EN: {s}\nFR: {r}\n")


def make_translation_prompt(
    src_text: str,
    src_name: str,
    tgt_name: str,
    model_family: str = "generic",
    thinking: bool = True,
) -> str:
    prompt = (
        f"Please write a high-quality {tgt_name} translation of the following {src_name} sentence\n"
        f"\"{src_text}\"\n"
        "Please provide only the translation, nothing more.\n"
    )

    # Paper's Qwen non-thinking mode
    if model_family.lower().startswith("qwen") and not thinking:
        prompt += "<think>\n\n</think>\n"

    return prompt


import re

def _extract_translation(decoded_text: str) -> str:
    if not decoded_text:
        return ""

    text = decoded_text.strip()

    # Remove thinking traces
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

    # Prefer content after "Final Translation" if present
    m = re.search(r"Final Translation\s*:?\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # Drop any remaining tags
    text = re.sub(r"</?[^>]+>", "", text).strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text.strip(" \"'")


def _build_chat_prompts(sources, prompt_fn, enable_thinking=False):
    prompts = []
    for src in sources:
        msgs = [{'role': 'user', 'content': prompt_fn(src)}]
        prompts.append(
            tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        )
    return prompts


def translate(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=256,
    enable_thinking=False,
    batch_size=1024,
):
    """
    Batched generation for much better GPU utilization on Colab.
    Falls back to a smaller batch automatically if CUDA OOM happens.
    """
    prompts = _build_chat_prompts(sources, prompt_fn, enable_thinking=enable_thinking)
    translations = []
    i = 0

    pbar = tqdm(total=len(prompts))
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)

        while current_bs >= 1:
            batch_prompts = prompts[i:i + current_bs]
            try:
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=None,
                        top_p=None,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                # IMPORTANT: generated tokens start after the padded input width,
                # not after each sample's non-pad length.
                prompt_width = inputs['input_ids'].shape[1]

                for row in outputs:
                    generated_tokens = row[prompt_width:]
                    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                    translations.append(_extract_translation(decoded))

                i += current_bs
                pbar.update(current_bs)
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                current_bs //= 2
                if current_bs == 0:
                    raise RuntimeError(
                        'Out of memory even with batch_size=1. '
                        'Lower max_new_tokens or use a smaller model.'
                    )

    pbar.close()
    return translations


def score(hyps, refs):
    bleu = evaluate.load('sacrebleu')
    chrf = evaluate.load('chrf')
    b = bleu.compute(predictions=hyps, references=[[r] for r in refs])['score']
    c = chrf.compute(predictions=hyps, references=[[r] for r in refs], word_order=2)['score']
    return round(b, 2), round(c, 2)

print('Utilities loaded! Batched translation is enabled.')

In [ ]:
# ── EXP 2: ZERO-SHOT CoT PROMPTING FOR MT WITH GEMMA-1B ─────────────────

import re
import json
import pandas as pd
from tqdm.notebook import tqdm

# -------------------------------------------------------------------------
# Exact CoT construction templates from Appendix A.3 of the paper
# -------------------------------------------------------------------------

COT_TEMPLATES = {
    "T1": """<think>
1. Analyze the sentence structure and identify the core elements (subject, verb, object).
2. Translate the sentence from the origin language to the target language, focusing on the core elements.
3. Review the translation for basic accuracy and grammatical structure.
4. Identify areas that need further refinement (e.g., word choice, tense, or word order).
5. Modify the translation to improve fluency and coherence, considering the context.
6. Finalize the translation by ensuring it retains the original meaning while improving readability.
</think>""",

    "T2": """<think>
1. Identify basic elements: Break down the sentence into its main components and identify the key subject, verb, and object.
2. Translate to intermediate language: Convert these elements into an intermediate language structure (e.g., simple syntactic rules or function names).
3. Refine back to target language: Translate from the intermediate language back to the target language, adjusting for syntactic norms and idiomatic expressions.
4. Check for accuracy: Ensure that the meaning is preserved in the translated sentence by checking noun-verb agreement and connectors.
5. Adjust word order: Modify word order to ensure that it aligns with the target language’s grammatical structure.
6. Final refinement: Review the translation for naturalness, idiomatic use, and overall flow.
</think>""",

    "T3": """<think>
1. Analyze the provided context in the source language.
2. Translate the source text to the target language.
3. Perform back translation from the target language to the source language.
4. Compare the back translation with the original source context.
5. Evaluate whether the meaning of the back translation aligns with the original.
6. If discrepancies are identified, adjust the target language translation to enhance consistency with the original meaning.
7. Finalize the translation by ensuring both forward and back translations accurately align across all languages involved.
</think>""",

    "T4": """<think>
1. Analyze the current sentence, along with the previous sentences, to understand the overall conversation context.
2. Identify key elements like tone, formality, or subject matter based on the ongoing conversation.
3. Translate the sentence while ensuring that the translation is aligned with the tone, style, and subject of the preceding dialogue.
4. If any ambiguity exists in the translation due to context, refine the translation to better fit the conversation flow.
5. Verify that the translation maintains coherence with the larger conversation, ensuring consistency in language and tone.
6. Finalize the translation by cross-checking it with the conversation’s context to ensure it feels natural and appropriately aligned.
</think>""",

    "T5": """<think>
1. Analyze the source sentence and identify the key elements (verbs, subjects, objects, etc.).
2. Based on these elements, determine the most suitable translation strategy (literal vs. idiomatic).
3. Select the best translation for each word or phrase, considering context and language-specific structures.
4. Explain the rationale behind choosing specific words or phrases.
5. After completing the initial translation, review each translation decision and explain any adjustments made for fluency or accuracy.
6. Provide a final explanation for the translation choices, discussing any trade-offs made between literal meaning and contextual appropriateness.
</think>""",

    "T6": """<think>
1. Analyze the sentence’s syntactic structure in the source language (e.g., identify whether it’s active or passive).
2. Determine the most appropriate syntactic structure in the target language (e.g., whether it needs to be rephrased from active to passive or vice versa).
3. Adjust the word order and grammatical structure in the target language to match the sentence’s meaning, while maintaining clarity.
4. Translate the sentence, ensuring that subject-verb-object relationships and other syntactic elements align with target language norms.
5. After the translation, check the sentence’s grammar and overall flow in the target language, making sure it is clear and fluid.
6. If the sentence feels awkward or unnatural, refine the structure by adjusting word choice or reordering components.
</think>""",
}

STRATEGY_LABELS = {
    "cot_zero_shot": "Let's think step by step",
    "T1": "T1 - Hierarchical Translation",
    "T2": "T2 - Triangulating Translation",
    "T3": "T3 - Back Translation",
    "T4": "T4 - Context-aware Translation",
    "T5": "T5 - Translation Explanation",
    "T6": "T6 - Structural Transformation",
}

# -------------------------------------------------------------------------
# Prompt builder
# -------------------------------------------------------------------------

def make_cot_translation_prompt(
    src_text: str,
    src_name: str,
    tgt_name: str,
    strategy: str = "cot_zero_shot",
) -> str:
    """
    Zero-shot CoT prompting for MT with explicit <think>...</think> markup
    plus 'Final Translation' marker for easy extraction.
    """

    base = (
        f"Please write a high-quality {tgt_name} translation of the following {src_name} sentence.\n\n"
        f"{src_name}: {src_text}\n\n"
    )

    if strategy == "cot_zero_shot":
        reasoning_instruction = (
            "Let's think step by step.\n"
            "First reason about the translation inside <think> and </think>.\n"
            "Then output the final translation after the line 'Final Translation'.\n"
            "Do not put anything except reasoning inside <think>...</think>.\n"
            "Do not put anything after the final translation.\n"
        )
    elif strategy in COT_TEMPLATES:
        reasoning_instruction = (
            "You should reason before translating.\n"
            "Write your reasoning inside <think> and </think>.\n"
            "Follow this Thinking Chain Guide as closely as possible:\n"
            f"{COT_TEMPLATES[strategy]}\n\n"
            "Then output the final translation after the line 'Final Translation'.\n"
            "Do not put anything except reasoning inside <think>...</think>.\n"
            "Do not put anything after the final translation.\n"
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    format_hint = (
        "Output format:\n"
        "<think>\n"
        "...\n"
        "</think>\n\n"
        "Final Translation\n"
        "<your translation here>"
    )

    return base + reasoning_instruction + "\n" + format_hint


# -------------------------------------------------------------------------
# Extraction utilities
# -------------------------------------------------------------------------

def _extract_translation_from_cot(decoded_text: str) -> str:
    """
    Extract final translation from outputs of the form:
      <think> ... </think>
      Final Translation
      ...
    Also robust to imperfect outputs.
    """
    if not decoded_text:
        return ""

    text = decoded_text.strip()

    # Prefer content after the last 'Final Translation'
    m = re.search(r"Final Translation\s*:?\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        candidate = m.group(1).strip()
        candidate = re.sub(r"</?[^>]+>", "", candidate).strip()
        candidate = re.sub(r"\s+", " ", candidate).strip()
        return candidate.strip(" \"'")

    # Fallback: remove the think block, then clean
    text_wo_think = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()
    text_wo_think = re.sub(r"</?[^>]+>", "", text_wo_think).strip()
    text_wo_think = re.sub(r"\s+", " ", text_wo_think).strip()

    return text_wo_think.strip(" \"'")


def _extract_think_block(decoded_text: str) -> str | None:
    if not decoded_text:
        return None
    m = re.search(r"<think>(.*?)</think>", decoded_text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else None


def _contains_think_block(decoded_text: str) -> bool:
    if not decoded_text:
        return False
    return re.search(r"<think>.*?</think>", decoded_text, flags=re.DOTALL | re.IGNORECASE) is not None


# -------------------------------------------------------------------------
# Raw-output translation helper
# Assumes you already have:
#   - _build_chat_prompts(...)
#   - model, tokenizer
# -------------------------------------------------------------------------

def translate_with_raw_outputs_cot(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=512,
    batch_size=16,
):
    prompts = _build_chat_prompts(
        sources,
        prompt_fn,
        enable_thinking=False,
    )

    raw_outputs = []
    hyps = []

    i = 0
    pbar = tqdm(total=len(prompts), desc="Generating", leave=False)
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)
        batch_prompts = prompts[i:i + current_bs]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

        prompt_width = inputs["input_ids"].shape[1]

        for row in outputs:
            generated_tokens = row[prompt_width:]
            raw_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            raw_outputs.append(raw_text)
            hyps.append(_extract_translation_from_cot(raw_text))

        i += current_bs
        pbar.update(current_bs)
    pbar.close()

    return raw_outputs, hyps


# -------------------------------------------------------------------------
# Experiment runner
# -------------------------------------------------------------------------

COT_STRATEGIES = ["cot_zero_shot", "T1", "T2", "T3", "T4", "T5", "T6"]

def run_exp2_cot_prompting(
    model,
    tokenizer,
    lang_pairs,
    n_eval=50,
    max_new_tokens=512,
    batch_size=16,
    n_examples_to_store=3,
    n_examples_to_print=1,
):
    exp2_results = []

    for src_lang, tgt_lang, src_name, tgt_name in tqdm(lang_pairs, desc="Language pairs"):
        print(f"\n{'='*100}")
        print(f"EXP 2 — CoT prompting with Gemma-1B: {src_name} → {tgt_name}")
        print(f"{'='*100}")

        sources, references = load_flores(src_lang, tgt_lang, n=n_eval)

        for strategy in tqdm(COT_STRATEGIES, desc="Strategies"):
            tqdm.write(f"Running strategy: {STRATEGY_LABELS[strategy]}")

            label = STRATEGY_LABELS[strategy]

            fn = lambda s, sn=src_name, tn=tgt_name, strat=strategy: make_cot_translation_prompt(
                s, sn, tn, strat
            )

            raw_outputs, hyps = translate_with_raw_outputs_cot(
                model=model,
                tokenizer=tokenizer,
                sources=sources,
                prompt_fn=fn,
                max_new_tokens=max_new_tokens,
                batch_size=batch_size,
            )

            bleu, chrf = score(hyps, references)
            think_flags = [_contains_think_block(x) for x in raw_outputs]
            think_ratio = sum(think_flags) / len(think_flags) if think_flags else 0.0

            print(
                f"{label:35s} | BLEU: {bleu:6.2f} | chrF++: {chrf:6.2f} | "
                f"with <think>: {100*think_ratio:5.1f}%"
            )

            examples = []
            for i, (src, ref, raw, hyp, has_think) in enumerate(
                zip(
                    sources[:n_examples_to_store],
                    references[:n_examples_to_store],
                    raw_outputs[:n_examples_to_store],
                    hyps[:n_examples_to_store],
                    think_flags[:n_examples_to_store],
                )
            ):
                examples.append({
                    "example_id": i,
                    "source": src,
                    "reference": ref,
                    "raw_output": raw,
                    "contains_think_block": has_think,
                    "think_block": _extract_think_block(raw),
                    "hypothesis": hyp,
                })

            if n_examples_to_print > 0:
                print(f"  Example output for {label}:")
                for ex in examples[:n_examples_to_print]:
                    print("  ---")
                    print("  SOURCE:", ex["source"])
                    print("  REFERENCE:", ex["reference"])
                    print("  RAW OUTPUT:", ex["raw_output"].replace("\n", " "))
                    print("  EXTRACTED:", ex["hypothesis"])

            exp2_results.append({
                "src": src_name,
                "tgt": tgt_name,
                "strategy": strategy,
                "strategy_label": label,
                "bleu": bleu,
                "chrf": chrf,
                "n_outputs": len(raw_outputs),
                "n_with_think_block": int(sum(think_flags)),
                "prop_with_think_block": think_ratio,
                "examples": examples,
            })

    return exp2_results

In [ ]:
# -------------------------------------------------------------------------
# EXPERIMENT 2
# -------------------------------------------------------------------------

GEMMA_MODEL_ID = "google/gemma-3-1b-it"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# -----------------------------
# Load Gemma-1B IT
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

# Example:
exp2_results = run_exp2_cot_prompting(
    model=model,
    tokenizer=tokenizer,
    lang_pairs=LANG_PAIRS,
    n_eval=50,
    max_new_tokens=512,
    batch_size=1024,
    n_examples_to_store=50,
    n_examples_to_print=3,
)

df2 = pd.DataFrame([
    {k: v for k, v in row.items() if k != "examples"}
    for row in exp2_results
])
print(df2.to_string(index=False))

with open("results_exp2_cot_prompting.json", "w", encoding="utf-8") as f:
    json.dump(exp2_results, f, indent=2, ensure_ascii=False)

df2.to_csv("results_exp2_cot_prompting.csv", index=False, encoding="utf-8")